# 26 — Tüm eğitilmiş modellerin consensus dış test değerlendirmesi

Bu notebook iki model grubunu aynı dış test kümelerinde değerlendirir:

1. Consensus eğitimli modeller: BERT, DistilBERT, RoBERTa ve FinBERT
2. İlk eğitimli modeller: BERT, DistilBERT, RoBERTa ve hedef veri setinde fine-tune edilmiş FinBERT

S&P 500 ve Reuters için yalnızca ChatGPT = Gemini olan kayıtlar kullanılır ve `consensus_label` gold etiketidir. Sentetik test setinde Gemini/ChatGPT anotasyonu bulunmadığı için mevcut `label` kolonu kullanılır.

Not: Burada consensus filtresi model eğitimine değil, dış test gold etiketinin oluşturulmasına uygulanmaktadır.

In [1]:
from pathlib import Path
import gc, json, re, sys
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report, confusion_matrix

PROJECT_ROOT = Path('D:/serkan.kaymak/financial_sentiment_thesis/financial_sentiment_thesis')
APP_ROOT = PROJECT_ROOT / 'app'
if str(APP_ROOT) not in sys.path: sys.path.insert(0, str(APP_ROOT))
from thesis_utils import paths

DATA_ROOT = PROJECT_ROOT / 'db'
CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT
RESULTS_ROOT = paths.MODEL_RESULTS_ROOT / 'all_models_consensus_external_evaluation'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABELS = list(LABEL2ID)
MAX_LENGTH, BATCH_SIZE = 128, 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_GROUPS = {
    'consensus_trained': {
        'bert': CHECKPOINT_ROOT / 'consensus_twitter_phrasebank_allagree_v1' / 'bert' / 'final_model',
        'distilbert': CHECKPOINT_ROOT / 'consensus_twitter_phrasebank_allagree_v1' / 'distilbert' / 'final_model',
        'roberta': CHECKPOINT_ROOT / 'consensus_twitter_phrasebank_allagree_v1' / 'roberta' / 'final_model',
        'finbert': CHECKPOINT_ROOT / 'consensus_twitter_phrasebank_allagree_v1' / 'finbert' / 'final_model',
    },
    'initial_trained': {
        'bert': CHECKPOINT_ROOT / 'bert_base_uncased' / 'final_model',
        'distilbert': CHECKPOINT_ROOT / 'distilbert_base_uncased' / 'final_model',
        'roberta': CHECKPOINT_ROOT / 'roberta_base' / 'final_model',
        'finbert': CHECKPOINT_ROOT / 'finbert_target_finetuned_seed42' / 'final_model',
    },
}
print('Device:', DEVICE)
print('Results:', RESULTS_ROOT)

Device: cpu
Results: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\outputs\finetuned_model_results\all_models_consensus_external_evaluation


In [2]:
def norm_series(s):
    return s.astype('string').str.strip().str.lower().replace({'': pd.NA, 'nan': pd.NA, 'none': pd.NA, '<na>': pd.NA})

def make_consensus_eval(df, dataset_name):
    df = df.copy()
    text_col = next((c for c in ['annotation_text', 'text_en', 'text', 'input_text', 'headline', 'title'] if c in df.columns), None)
    if text_col is None: raise ValueError(f'{dataset_name}: metin kolonu bulunamadı')
    df['eval_text'] = norm_series(df[text_col])
    if not {'chatgpt_label', 'gemini_label'}.issubset(df.columns): raise ValueError(f'{dataset_name}: consensus kolonları eksik')
    chat = norm_series(df['chatgpt_label']); gem = norm_series(df['gemini_label'])
    agreement = (chat == gem) & chat.notna() & gem.notna()
    if not agreement.all(): raise ValueError(f'{dataset_name}: ChatGPT-Gemini eşleşmeyen veya eksik consensus satırı bulundu')
    if 'consensus_label' in df.columns:
        stored = norm_series(df['consensus_label'])
        if not (stored == chat).fillna(False).all(): raise ValueError(f'{dataset_name}: consensus_label, ChatGPT-Gemini etiketiyle uyuşmuyor')
    else:
        df['consensus_label'] = chat
    df['gold_label'] = chat
    df = df[df['eval_text'].notna() & df['gold_label'].isin(LABELS)].copy().reset_index(drop=True)
    df['gold_id'] = df['gold_label'].map(LABEL2ID).astype(int)
    df.insert(0, 'sample_id', [f'{dataset_name}_{i:06d}' for i in range(len(df))]) if 'sample_id' not in df.columns else None
    df['dataset'] = dataset_name
    return df

def read_synthetic():
    files = sorted((DATA_ROOT / 'evaluation' / 'synthetic_financial_news').glob('*_batch_*.xlsx'))
    frames = [pd.read_excel(p) for p in files if not p.name.startswith('~$')]
    df = pd.concat(frames, ignore_index=True)
    df['consensus_label'] = norm_series(df['label'])
    return make_consensus_eval(df, 'synthetic_external')

sp500_path = DATA_ROOT / 'annotations' / 'additional_5000_2026' / 'final' / 'sp500_additional_5000_consensus.csv'
reuters_path = DATA_ROOT / 'annotations' / 'additional_5000_2026' / 'final' / 'reuters_additional_5000_consensus.csv'
sp500 = make_consensus_eval(pd.read_csv(sp500_path), 'sp500_consensus_external')
reuters = make_consensus_eval(pd.read_csv(reuters_path), 'reuters_consensus_external')
synthetic = read_synthetic()
EVAL_SETS = {'sp500_consensus_external': sp500, 'synthetic_external': synthetic, 'reuters_consensus_external': reuters}
print({k: {'n': len(v), 'labels': v['gold_label'].value_counts().to_dict()} for k, v in EVAL_SETS.items()})
assert len(sp500) == 4510, len(sp500)
assert len(reuters) == 3848, len(reuters)
assert len(synthetic) == 3000, len(synthetic)

{'sp500_consensus_external': {'n': 4510, 'labels': {'neutral': 2095, 'positive': 1393, 'negative': 1022}}, 'synthetic_external': {'n': 3000, 'labels': {'negative': 1000, 'neutral': 1000, 'positive': 1000}}, 'reuters_consensus_external': {'n': 3848, 'labels': {'negative': 1780, 'positive': 1132, 'neutral': 936}}}


In [3]:
def predict(model_dir, texts):
    if not model_dir.exists(): raise FileNotFoundError(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    model.eval(); rows = []
    with torch.no_grad():
        for start in tqdm(range(0, len(texts), BATCH_SIZE), desc=model_dir.parent.parent.name + '/' + model_dir.parent.name):
            batch = texts[start:start+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            rows.append(torch.softmax(model(**enc).logits, dim=-1).cpu().numpy())
    del model, tokenizer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(rows)

def evaluate(group_name, model_key, model_dir, dataset_name, df):
    out = RESULTS_ROOT / group_name / model_key / dataset_name; out.mkdir(parents=True, exist_ok=True)
    probs = predict(model_dir, df['eval_text'].astype(str).tolist())
    pred = probs.argmax(axis=1); y = df['gold_id'].to_numpy()
    precision, recall, f1, support = precision_recall_fscore_support(y, pred, labels=[0,1,2], zero_division=0)
    metrics = {'training_group': group_name, 'model_key': model_key, 'dataset': dataset_name, 'gold_source': 'consensus_label' if 'consensus' in dataset_name else 'label', 'n_eval': len(df), 'accuracy': accuracy_score(y,pred), 'precision_macro': precision.mean(), 'recall_macro': recall.mean(), 'f1_macro': f1.mean(), 'f1_weighted': f1_score(y,pred,average='weighted'), 'f1_negative': f1[0], 'f1_neutral': f1[1], 'f1_positive': f1[2], 'support_negative': int(support[0]), 'support_neutral': int(support[1]), 'support_positive': int(support[2])}
    pred_df = df.copy(); pred_df['predicted_label'] = [ID2LABEL[int(i)] for i in pred]; pred_df['confidence'] = probs.max(axis=1)
    for i, label in ID2LABEL.items(): pred_df[f'prob_{label}'] = probs[:, i]
    pred_df.to_csv(out / 'predictions.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(classification_report(y, pred, labels=[0,1,2], target_names=LABELS, output_dict=True, zero_division=0)).T.to_csv(out / 'classification_report.csv', encoding='utf-8-sig')
    cm = confusion_matrix(y, pred, labels=[0,1,2]); pd.DataFrame(cm, index=[f'true_{x}' for x in LABELS], columns=[f'pred_{x}' for x in LABELS]).to_csv(out / 'confusion_matrix.csv', encoding='utf-8-sig')
    (out / 'metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8')
    return metrics

results = []
for group_name, models in MODEL_GROUPS.items():
    for model_key, model_dir in models.items():
        for dataset_name, df in EVAL_SETS.items():
            print(f'{group_name} | {model_key} | {dataset_name} | n={len(df)}')
            results.append(evaluate(group_name, model_key, model_dir, dataset_name, df))
            gc.collect()
summary = pd.DataFrame(results)
summary.to_csv(RESULTS_ROOT / 'all_models_consensus_external_summary.csv', index=False, encoding='utf-8-sig')
display(summary[['training_group','model_key','dataset','n_eval','accuracy','f1_macro','f1_weighted']].round(4))

consensus_trained | bert | sp500_consensus_external | n=4510


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

consensus_twitter_phrasebank_allagree_v1/bert:   0%|          | 0/71 [00:00<?, ?it/s]

consensus_trained | bert | synthetic_external | n=3000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

consensus_twitter_phrasebank_allagree_v1/bert:   0%|          | 0/47 [00:00<?, ?it/s]

... (63 outputs left out)

initial_trained | finbert | reuters_consensus_external | n=3848


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

financial_sentiment_multi_model/finbert_target_finetuned_seed42:   0%|          | 0/61 [00:00<?, ?it/s]

,training_group,model_key,dataset,n_eval,accuracy,f1_macro,f1_weighted
0,consensus_trained,bert,sp500_consensus_external,4510,0.8129,0.8045,0.8115
1,consensus_trained,bert,synthetic_external,3000,0.9817,0.9816,0.9816
2,consensus_trained,bert,reuters_consensus_external,3848,0.7199,0.7069,0.7262
3,consensus_trained,distilbert,sp500_consensus_external,4510,0.7956,0.7856,0.7946
4,consensus_trained,distilbert,synthetic_external,3000,0.9687,0.9686,0.9686
5,consensus_trained,distilbert,reuters_consensus_external,3848,0.6920,0.6819,0.6987
6,consensus_trained,roberta,sp500_consensus_external,4510,0.8583,0.8551,0.8578
7,consensus_trained,roberta,synthetic_external,3000,0.9953,0.9953,0.9953
8,consensus_trained,roberta,reuters_consensus_external,3848,0.7633,0.7500,0.7682
9,consensus_trained,finbert,sp500_consensus_external,4510,0.8248,0.8173,0.8228


## Çıktılar

Sonuçlar `outputs/finetuned_model_results/all_models_consensus_external_evaluation/` altında kaydedilir. Her model ve veri seti için tahminler, classification report, confusion matrix ve metrics JSON oluşturulur. Özet dosya: `all_models_consensus_external_summary.csv`.